# Taller de la semana 1 — SOLUCIÓN DE REFERENCIA (solo para el docente)

**Enunciado:** [`README.md`](README.md) · **Plan de clase:** [`../../weeks/week-01.md`](../../weeks/week-01.md) · **Motor proporcionado:** [`eliza.py`](eliza.py)

Este notebook presenta la **explicación didáctica** del taller de la semana 1.
**Importa el código de referencia de [`solution.py`](solution.py)** sin volver a copiarlo,
por lo que la solución mostrada es exactamente la que evalúan las pruebas proporcionadas.
La garantía de corrección se comprueba por separado: el comando documentado
`python3 studios/_verify_solutions.py week-01` ejecuta `test_eliza.py` sin modificaciones
contra `solution.py`.

> No distribuir. Se excluye de la versión para estudiantes mediante `studios/.gitignore`.


In [1]:
# --- Preparación: añade la carpeta de la semana (solution/eliza) y la raíz del repositorio
# (aicourse) a la ruta de importación para ejecutar desde las ubicaciones previstas. ---
import sys, pathlib
# POO: pathlib.Path es una clase; cwd() devuelve un objeto que representa una ruta.
# El operador / construye rutas y exists() consulta si el archivo existe.
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-01"
root = week.parent.parent
for p in (str(week), str(root)):
    # sys.path es una lista; insert(0, p) añade una ruta con prioridad de búsqueda.
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect, random
import eliza
import solution

## Tarea 1a — Tres reglas nuevas

El requisito: **al menos 3** reglas, cada una con un disparador *real* (ningún patrón
puede coincidir con la cadena vacía, porque eclipsaría las respuestas por defecto),
y **ninguna** debe duplicar un patrón base. Este es el conjunto de referencia,
obtenido directamente de `solution.py`:


In [2]:
# getsource obtiene el código de la función importada para mostrarlo sin duplicarlo.
print(inspect.getsource(solution.new_rules))
# Un conjunto guarda los patrones base; _ ignora las plantillas de cada tupla.
base = {p for p, _ in eliza.RULES}
for pat, templates in solution.new_rules():
    import re
    # Las aserciones comprueban que el patrón no acepte vacío ni duplique uno base.
    assert re.search(pat, "") is None, "would fire on everything"
    assert pat not in base, "duplicates a base rule"
    print("novel & real-trigger:", pat)

def new_rules():
    """Three novel rules. Each has a real trigger (none matches the empty
    string) and none duplicates a base pattern in ``eliza.RULES``."""
    return [
        (r"\bI want (.*)", ["What would it mean to you to get {0}?",
                            "Why do you want {0}?"]),
        (r"\bI can't (.*)", ["What makes you feel you can't {0}?",
                             "Have you ever been able to {0}?"]),
        (r"\bI think (.*)", ["Do you really think {0}?",
                             "What led you to think {0}?"]),
    ]

novel & real-trigger: \bI want (.*)
novel & real-trigger: \bI can't (.*)
novel & real-trigger: \bI think (.*)


## Tarea 1b — La función de respuesta ampliada

Las reglas base **tienen prioridad** para que las nuevas no alteren la demostración;
luego se prueban las nuevas y finalmente se usa una respuesta por defecto. Es el
mismo bucle de `eliza.respond` sobre la tabla combinada: *reutilizamos* `reflect`
del motor, sin volver a implementarla.


In [3]:
print(inspect.getsource(solution.extended_respond))
# La semilla permite repetir la selección aleatoria de plantillas.
random.seed(1)
# La regla base sigue teniendo prioridad: el requisito central de la tarea 1b.
reply = solution.extended_respond("I feel anxious most of the time")
print("base-rule reply :", reply)
# lower() y startswith() son métodos de str: normalizan y comprueban el inicio.
assert reply.lower().startswith("do you often feel") and "anxious" in reply.lower()
# Una regla nueva se activa cuando ninguna regla base coincide.
# La semilla permite repetir la selección aleatoria de plantillas.
random.seed(1)
print("new-rule reply  :", solution.extended_respond("I want to pass this course"))

def extended_respond(text):
    """Base rules FIRST (so the demo never regresses), then ``new_rules()``,
    then the fallback. Same loop as ``eliza.respond``, over the joined table."""
    for pattern, templates in RULES + new_rules():
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            groups = [reflect(g) for g in m.groups()]
            return random.choice(templates).format(*groups)
    return random.choice(FALLBACK)

base-rule reply : Do you often feel anxious most of the time?
new-rule reply  : What would it mean to you to get to pass this course?


## Tarea 1c — El fallo deliberado: observa cómo falla la garantía

Este es el objetivo central de la semana 1. El fallo declarado es una **oración compuesta**
cuya *segunda cláusula se omite por completo*: al ganar la primera coincidencia,
ELIZA nunca la examina. A continuación reproducimos exactamente lo que comprueba
`test_declared_failure_is_hollow`: ninguna palabra de contenido de la cláusula B
aparece en la respuesta.


In [4]:
import re
# Desempaqueta la tupla: text contiene la entrada y why explica el fallo.
text, why = solution.failure_case()
random.seed(1)
reply = solution.extended_respond(text)
print("input :", text)
print("reply :", reply)
print("why   :", why)

# split divide solo una vez; _ descarta la cláusula A y clause_b conserva la B.
_, clause_b = text.lower().split(" and ", 1)
# Una comprensión de lista selecciona las palabras de al menos cuatro letras.
content = [w for w in re.findall(r"[a-z]+", clause_b) if len(w) >= 4]
surfaced = [w for w in content if w in reply.lower()]
print("\nclause B content words:", content)
print("surfaced in the reply :", surfaced, "  <-- empty = the guarantee failed, as designed")
# Lista vacía significa que no reapareció ninguna de esas palabras: el fallo esperado.
assert surfaced == [], "clause B leaked; this would not be an ELIZA-class drop"
print("\nGUARANTEE FAIL confirmed: ELIZA answered fluently while ignoring half the sentence.")

input : My mother is a doctor and my father is a lawyer
reply : Tell me more about your mother.
why   : First-match-wins: the 'my (mother|father|family)' rule fires on clause A, so clause B ('my father is a lawyer') is never inspected — fluent, on-topic, and half-blind.

clause B content words: ['father', 'lawyer']
surfaced in the reply : []   <-- empty = the guarantee failed, as designed

GUARANTEE FAIL confirmed: ELIZA answered fluently while ignoring half the sentence.


## Tarea 2 — Primer contacto: ELIZA frente a un modelo real

Los estudiantes envían los **mismos** prompts a ELIZA y a un modelo local, registran
ambas transcripciones y responden las tres preguntas del README. Aquí ejecutamos
la comparación sin interacción con el backend determinista `echo`, para que el
notebook pueda ejecutarse en cualquier entorno. **Para una comparación real,
vuelve a ejecutarlo con `backend="ollama"` o `backend="manual"`**: `echo` es un
simulador y su texto nunca debe reportarse como resultado de un modelo.

La mala práctica que hay que señalar: **no pidas al modelo que se evalúe a sí mismo.**
Tú eres quien verifica (eje 3 de la ficha de evaluación).


In [5]:
import os, json
from aicourse.llm import LLM

# json.load convierte el JSON en un diccionario; first_contact contiene una lista.
# Sus cadenas se conservan en inglés porque ELIZA busca patrones en ese idioma.
prompts = json.load(open(week / "prompts.json"))["first_contact"]
backend = os.environ.get("AICOURSE_NB_BACKEND", "echo")   # echo = simulador que no requiere interacción
# POO: LLM es la clase; llm es una instancia creada con esta configuración.
llm = LLM(backend=backend)

random.seed(1)
# El corte [:4] toma cuatro entradas para esta demostración.
for p in prompts[:4]:
    print("you  >", p)
    print("ELIZA>", solution.extended_respond(p))
    # complete es un método de llm; el objeto devuelto tiene el atributo text.
    print("LLM  >", llm.complete(p).text)
    print()

you  > I am not feeling great about the exam
ELIZA> How long have you been not feeling great about the exam?
LLM  > [echo backend] 37 chars received. This is NOT a model response.

you  > I feel anxious most of the time
ELIZA> Do you often feel anxious most of the time?
LLM  > [echo backend] 31 chars received. This is NOT a model response.

you  > My mother thinks I should study more
ELIZA> Tell me more about your mother.
LLM  > [echo backend] 36 chars received. This is NOT a model response.

you  > I did badly because I did not sleep
ELIZA> Is that the real reason?
LLM  > [echo backend] 35 chars received. This is NOT a model response.



### Qué deben escribir los estudiantes

1. ¿En qué casos ELIZA pareció *más* honesta que el modelo (reconoció sus límites frente a fingir que sabía)?
2. El modelo no tiene más *comprensión* que ELIZA: ¿de qué tiene *más* y por qué eso hace que sus fallos sean **más difíciles** de predecir?
3. Menciona una entrada para la que puedas **demostrar** qué hará ELIZA y otra para la que no puedas demostrar qué hará el modelo. Esa diferencia es el tema central del curso.

Volverán a leer su respuesta de la semana 1 en la semana 14.
